In [1]:
# Import necessary modules
import os
import re
import glob
from utils import *
import contextily as ctx
import utils.for_setpath as path
import utils.for_empatica as empatica

# Data processing
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# Pluma-python API  
from modules import *
from pluma.schema.outdoor import build_schema
import schemas.missing_sync as missing_sync
from statsmodels.stats.anova import AnovaRM

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [4]:
# read access only
rawdata = "/mnt/raid/emotional_data_raquel"
# where im adding data to
workdir = "/home/s243636/master-thesis"

# GLM

The goal here is to model the variability in physiological data by the variability of the outdoor stimuli, which include climate, geographical and social stimuli.
A GLM will be fitted to the data. The design matrix will include rows corresponding to averages of data from 10 meter intervals. This increases SNR.

This function aggregates data into fixed spatial intervals by averaging all measurements within consecutive distance segments, enabling spatially consistent analysis of physiological and environmental variables.

In [5]:
def average_by_gps(df, dist, plot=False, output_path=None, column_order=None):
    """
    Averages rows in the DataFrame based on the 'cum_dist' column.
    All rows whose 'cum_dist' falls into the same interval (of length `dist` meters)
    are averaged together. Optionally, the function plots a basemap of the averaged GPS
    intervals and filters the averaged DataFrame to only include a given set of columns.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing a "cum_dist" column (in meters) along with GPS and other numeric data.
    dist : float
        Distance interval in meters over which to average the data.
    plot : bool, optional
        If True, the function will generate a basemap plot showing the averaged intervals.
    output_path : str, optional
        Directory in which to save the plot if plot==True. If None, the plot is displayed.
    column_order : list, optional
        A list of column names to keep in the final averaged DataFrame.
        Only those columns from this list that exist in the averaged data will be kept.
    
    Returns:
    --------
    avg_df : pd.DataFrame
        A new DataFrame with one row per interval (with bin midpoints) and averaged values.
        The averaged cumulative distance is stored in a column named "avg_cum_dist".
    """
    # Ensure we work on a copy
    df = df.copy()
    
    if 'cum_dist' not in df.columns:
        raise ValueError("Input dataframe must contain a 'cum_dist' column.")
    
    # Create bins from 0 up to maximum cum_dist, with step = dist
    max_dist = df['cum_dist'].max()
    bins = np.arange(0, max_dist + dist, dist)
    labels = (bins[:-1] + bins[1:]) / 2  # use the midpoints as labels
    df['dist_bin'] = pd.cut(df['cum_dist'], bins=bins, labels=labels, include_lowest=True)
    print(f"[INFO] Created {len(labels)} intervals from 0 to {max_dist:.2f} m (interval = {dist} m).")
    
    # Group by the distance bin and compute the mean of numeric columns.
    avg_df = df.groupby('dist_bin').mean(numeric_only=True).reset_index()
    # Rename the averaged cum_dist to 'avg_cum_dist'
    avg_df.rename(columns={'cum_dist': 'avg_cum_dist'}, inplace=True)
    
    # Optionally filter the resulting DataFrame to only include columns in column_order.
    if column_order is not None:
        available_columns = [col for col in column_order if col in avg_df.columns]
        print(f"[INFO] Available predictor columns in the averaged data: {available_columns}")
        # Keep the bin information as well
        avg_df = avg_df[['dist_bin'] + available_columns]
    
    # If plotting is requested, plot the averaged GPS intervals on a basemap.
    if plot:
        # Check that averaged data has the required GPS coordinate columns.
        if 'longitude_corrected' in avg_df.columns and 'latitude_corrected' in avg_df.columns:
            # Create a GeoDataFrame from the averaged data.
            gdf_avg = gpd.GeoDataFrame(avg_df, geometry=gpd.points_from_xy(avg_df.longitude_corrected, avg_df.latitude_corrected), crs="EPSG:4326")
            # Convert to Web Mercator for basemap plotting.
            gdf_web = gdf_avg.to_crs(epsg=3857)
            
            # Define a helper function to add a basemap using Contextily.
            def add_basemap_to_ax(ax, data_crs="EPSG:4326", source=None):
                if source is None:
                    source = ctx.providers.CartoDB.Positron
                ctx.add_basemap(ax, crs=data_crs, source=source)
            
            fig, ax = plt.subplots(figsize=(10, 6))
            # Plot the averaged points.
            gdf_web.plot(ax=ax, color='red', markersize=30, alpha=0.7)
            # Add basemap (here we pass data_crs as EPSG:3857 because gdf_web is in that projection).
            try:
                add_basemap_to_ax(ax, data_crs="EPSG:3857")
            except Exception as e:
                print("[WARN] Basemap skipped due to PROJ issue")
            # Remove grid lines and spines for a clean look.
            ax.grid(False)
            for spine in ax.spines.values():
                spine.set_visible(False)
            ax.set_title("Averaged GPS Data Intervals", fontsize=14)
            ax.set_xlabel("Longitude", fontsize=12)
            ax.set_ylabel("Latitude", fontsize=12)
            plt.tight_layout()
            if output_path is not None:
                if not os.path.exists(output_path):
                    os.makedirs(output_path)
                plot_file = os.path.join(output_path, "average_by_gps.png")
                plt.savefig(plot_file, dpi=300, bbox_inches='tight')
                print(f"[INFO] Basemap plot saved to {plot_file}")
                plt.close(fig)
            else:
                plt.show()
        else:
            print("[WARN] 'longitude' and/or 'latitude' columns not found in averaged data; skipping basemap plot.")
    
    return avg_df

# Example predictor list
column_order = [
    'time',
    'longitude_corrected', 'latitude_corrected',
    'original_longitude', 'original_latitude', 'elevation',
    'day_of_year', 'day', 'month', 'hour', 'minute', 'second',
    'participant_id', 'session_id', 'age', 'sex',
    'utci_stress_category',
    'typology',
    'valence', 'arousal', 'naturalness', 'crowdedness',
    'air_quality_index', 'air_temperature', 'air_humidity', 'air_pressure', 'sound_pressure_level',
    'humidity_sensor', 'analog_voltage', 'pm1_0', 'pm2_5', 'pm10_0', 'solar_light', 'thermocouple_temp',
    'ptc_air_temp', 'north_wind', 'east_wind', 'gust_wind', 'atmospheric_temperature', 'temp_atmospheric',
    'temp_tk', 'temp_tk_ptc', 'radiant_temp', 'Atmosferic Pressure', 'Dew Point',
    'empatica_heart_rate', 'skin_temperature', 'eda_raw', 'eda_phasic', 'x_acceleration', 'y_acceleration', 'z_acceleration', 'acceleration_magnitude', 'bvp',
    'delta', 'theta', 'alpha', 'beta', 'gamma', 'frontal alpha', 'frontal midline theta', 'theta-beta ratio', 'frontal alpha asymmetry', 'frontal theta'
]

my_dataframe = pd.read_csv(r"/home/s243636/master-thesis/fulldata/sub-OE002/ses-Hellerup/alldata.csv")
averaged_df = average_by_gps(my_dataframe, 10, plot=True, output_path=os.path.join(workdir, "fulldata_analysis", "glm"), column_order=column_order)

[INFO] Created 142 intervals from 0 to 1414.90 m (interval = 10 m).
[INFO] Available predictor columns in the averaged data: ['longitude_corrected', 'latitude_corrected', 'original_longitude', 'original_latitude', 'elevation', 'day_of_year', 'day', 'month', 'hour', 'minute', 'second', 'age', 'utci_stress_category', 'valence', 'arousal', 'naturalness', 'crowdedness', 'air_quality_index', 'air_temperature', 'air_humidity', 'air_pressure', 'sound_pressure_level', 'humidity_sensor', 'analog_voltage', 'pm1_0', 'pm2_5', 'pm10_0', 'solar_light', 'thermocouple_temp', 'ptc_air_temp', 'north_wind', 'east_wind', 'gust_wind', 'atmospheric_temperature', 'temp_atmospheric', 'temp_tk', 'temp_tk_ptc', 'radiant_temp', 'Atmosferic Pressure', 'Dew Point', 'empatica_heart_rate', 'skin_temperature', 'eda_raw', 'eda_phasic', 'x_acceleration', 'y_acceleration', 'z_acceleration', 'acceleration_magnitude', 'bvp']
[WARN] Basemap skipped due to PROJ issue


ERROR 1: PROJ: internal_proj_create_from_database: /home/s243636/miniforge3/envs/tese/share/proj/proj.db contains DATABASE.LAYOUT.VERSION.MINOR = 2 whereas a number >= 3 is expected. It comes from another PROJ installation.


[INFO] Basemap plot saved to /home/s243636/master-thesis/fulldata_analysis/glm/average_by_gps.png


In [6]:
averaged_df

,dist_bin,longitude_corrected,latitude_corrected,original_longitude,original_latitude,elevation,day_of_year,day,month,hour,...,Dew Point,empatica_heart_rate,skin_temperature,eda_raw,eda_phasic,x_acceleration,y_acceleration,z_acceleration,acceleration_magnitude,bvp
0,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,15.0,12.567827,55.730719,12.567745,55.730714,47506.812069,179.0,27.0,6.0,6.0,...,15.714774,153.592883,29.208017,0.015976,-0.000020,-18.366020,-61.343570,1.028376,64.898315,-0.030259
2,25.0,12.567836,55.730672,12.567743,55.730666,46688.475000,179.0,27.0,6.0,6.0,...,15.712217,NaN,29.264375,0.020982,0.000272,-17.015625,-64.335938,0.425781,67.042489,-0.027694
3,35.0,12.567855,55.730578,12.567762,55.730572,46999.587500,179.0,27.0,6.0,6.0,...,15.593131,NaN,29.266875,0.020141,0.000062,-17.195312,-63.167969,0.359375,66.014953,-0.141941
4,45.0,12.567876,55.730480,12.567823,55.730471,47450.688889,179.0,27.0,6.0,6.0,...,15.421252,NaN,29.285556,0.020822,0.000036,-17.699597,-62.763105,0.708109,65.663018,-0.004425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,1375.0,12.585066,55.728576,12.585068,55.728590,39539.057143,179.0,27.0,6.0,7.0,...,17.027693,NaN,28.410000,12.865794,0.405986,-26.165179,8.522321,57.763393,64.176804,-1.556193
138,1385.0,12.585173,55.728570,12.585175,55.728579,39361.071795,179.0,27.0,6.0,7.0,...,16.702641,NaN,28.389231,34.941331,-0.033051,-20.840545,-36.195513,24.710737,66.230481,0.313505
139,1395.0,12.585389,55.728557,12.585392,55.728574,38455.771429,179.0,27.0,6.0,7.0,...,15.602084,NaN,28.386429,35.598966,0.040653,-16.611607,-63.732143,1.901786,66.225531,0.060221
140,1405.0,12.585545,55.728548,12.585550,55.728569,38233.083333,179.0,27.0,6.0,7.0,...,15.646364,NaN,28.386667,35.292833,0.018190,-16.541667,-62.640625,0.328125,64.805676,-0.750055


In [7]:
def average_by_gps(df, dist):
    """
    Averages rows in the DataFrame based on the 'cum_dist' column.
    All rows whose cum_dist falls into the same interval of length 'dist'
    are averaged together.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing a "cum_dist" column (in meters) and other numeric columns.
    dist : float
        Distance interval in meters.
    
    Returns:
    --------
    pd.DataFrame
        A new DataFrame with one row per interval, where each column is averaged.
        Also includes an "avg_cum_dist" column that is the mean cum_dist in the interval.
    
    Note:
    -----
    The pandas groupby().mean(numeric_only=True) function ignores NaN values by default.
    """
    # Create bins from 0 to the maximum cumulative distance.
    max_dist = df['cum_dist'].max()
    bins = np.arange(0, max_dist + dist, dist)
    # Use the midpoints as labels.
    labels = (bins[:-1] + bins[1:]) / 2

    df = df.copy()
    df['dist_bin'] = pd.cut(df['cum_dist'], bins=bins, labels=labels, include_lowest=True)

    # Group by the distance bin and compute the mean for numeric columns.
    avg_df = df.groupby('dist_bin').mean(numeric_only=True).reset_index()
    # Rename the averaged 'cum_dist' to 'avg_cum_dist'
    avg_df.rename(columns={'cum_dist': 'avg_cum_dist'}, inplace=True)
    avg_df['dist_bin'] = avg_df['dist_bin'].astype(float)
    
    print(f"[INFO] Averaged data over {len(labels)} intervals using mean() which ignores NaN values.")
    return avg_df

def process_all_datasets(parent_dir, dist):
    """
    Process all datasets in a folder tree. For each CSV file found under the folder
    structure (e.g., sub-OE001/ses-Belem/alldata.csv), this function reads the CSV,
    calls average_by_gps with the specified distance interval, and adds columns extracted
    from the folder names (e.g., subject code and session name).
    
    The processed DataFrames are concatenated into one design matrix DataFrame.
    
    Parameters:
    -----------
    parent_dir : str
        The parent directory containing the dataset folders.
    dist : float
        The distance interval (in meters) to average over.
    
    Returns:
    --------
    pd.DataFrame
        A design matrix DataFrame where each row corresponds to the average data over a
        distance interval from one session.
    """
    all_dfs = []
    pattern = os.path.join(parent_dir, "sub-*", "ses-*", "alldata.csv")
    file_list = glob.glob(pattern, recursive=True)
    print(f"[INFO] Found {len(file_list)} dataset files.")
    
    for file_path in file_list:
        try:
            df = pd.read_csv(file_path)
            if 'cum_dist' not in df.columns:
                print(f"[WARN] Skipping {file_path}: 'cum_dist' column not found.")
                continue

            avg_df = average_by_gps(df, dist)
            m = re.search(r"sub-([A-Z0-9]+)", file_path, re.IGNORECASE)
            subject_code = m.group(1) if m else "Unknown"
            m2 = re.search(r"ses-([A-Za-z0-9]+)", file_path)
            session = m2.group(1) if m2 else "Unknown"
            avg_df['subject_code'] = subject_code
            avg_df['session'] = session
            avg_df['source_file'] = os.path.basename(file_path)
            all_dfs.append(avg_df)
        except Exception as e:
            print(f"[ERROR] Error processing {file_path}: {e}")
    
    if not all_dfs:
        print("[ERROR] No data processed.")
        return None
    design_df = pd.concat(all_dfs, ignore_index=True)
    print(f"[INFO] Design matrix has {design_df.shape[0]} rows and {design_df.shape[1]} columns.")
    return design_df

def fit_glm(design_df, response_var, predictors):
    """
    Fits a Generalized Linear Model (GLM) to the design matrix.
    
    Parameters:
    -----------
    design_df : pd.DataFrame
        The design matrix DataFrame with averaged rows.
    response_var : str
        The name of the dependent variable in the design matrix.
    predictors : list of str
        A list of column names to use as independent variables.
    
    Returns:
    --------
    model_fit : statsmodels.genmod.generalized_linear_model.GLMResultsWrapper
        The fitted GLM results.
    """
    # Columns we need to check for missing values
    cols_to_check = [response_var] + predictors
    total_rows = design_df.shape[0]
    print("[DEBUG] Missing values for GLM columns:")
    for col in cols_to_check:
        if col in design_df.columns:
            n_missing = design_df[col].isna().sum()
            pct_missing = (n_missing / total_rows) * 100
            print(f"  {col}: {n_missing} missing out of {total_rows} rows ({pct_missing:.1f}%)")
        else:
            print(f"  {col}: Column not found!")
    
    # Drop rows with missing values in the required columns
    design_df_clean = design_df.dropna(subset=cols_to_check)
    print(f"[INFO] GLM: {design_df_clean.shape[0]} rows remain after dropping missing values.")
    
    # Check if any rows remain
    if design_df_clean.shape[0] == 0:
        raise ValueError("No rows remain after dropping missing values; please check the input columns.")
    
    X = design_df_clean[predictors]
    X = sm.add_constant(X)
    y = design_df_clean[response_var]
    
    glm_model = sm.GLM(y, X, family=sm.families.Gaussian())
    model_fit = glm_model.fit()
    print(model_fit.summary())
    
    # Plot diagnostic plots
    # 1. Residual vs. Fitted
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(model_fit.fittedvalues, model_fit.resid_response, alpha=0.7, edgecolor='k')
    ax.set_xlabel("Fitted Values", fontsize=12)
    ax.set_ylabel("Residuals", fontsize=12)
    ax.set_title("Residual vs. Fitted Values", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # 2. QQ Plot of Residuals
    fig = sm.qqplot(model_fit.resid_response, line='45', fit=True)
    plt.title("QQ Plot of Residuals", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return model_fit


# Example usage:
if __name__ == '__main__':
    parent_directory = os.path.join(workdir, 'fulldata')
    distance_interval = 10
    design_matrix = process_all_datasets(parent_directory, distance_interval)
    
    # Define response and predictor columns.
    response = 'alpha'
    predictors = [
        'age', 'sex',
        # 'utci',
        # 'typology',
        # 'valence', 'arousal', 'naturalness', 'crowdedness',
        'wind_speed', 'noise_level', 'humidity', 'pm2_5', 'pm10_0', 'solar_light', 'temp_atmospheric',
    ]
    # Only keep predictors that are present in the design matrix.
    available_predictors = [p for p in predictors if p in design_matrix.columns]
    print(f"[INFO] Using predictors: {available_predictors}")
    
    if response in design_matrix.columns and available_predictors:
        glm_results = fit_glm(design_matrix, response, available_predictors)
        
        # GLM Diagnostic Plots:
        # 1. Residual vs. Fitted Plot.
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.scatter(glm_results.fittedvalues, glm_results.resid_response, alpha=0.7, edgecolor='k')
        ax.set_xlabel("Fitted Values", fontsize=12)
        ax.set_ylabel("Residuals", fontsize=12)
        ax.set_title("Residual vs. Fitted Values", fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # 2. QQ Plot of Residuals.
        fig = sm.qqplot(glm_results.resid_response, line='45', fit=True)
        plt.title("QQ Plot of Residuals", fontsize=14)
        plt.tight_layout()
        plt.show()
    else:
        print("[ERROR] The required columns for GLM are not present in the design matrix.")


[INFO] Found 25 dataset files.
[INFO] Averaged data over 109 intervals using mean() which ignores NaN values.
[INFO] Averaged data over 138 intervals using mean() which ignores NaN values.
[WARN] Skipping /home/s243636/master-thesis/fulldata/sub-OE024/ses-Nordhavn/alldata.csv: 'cum_dist' column not found.
[INFO] Averaged data over 136 intervals using mean() which ignores NaN values.
[WARN] Skipping /home/s243636/master-thesis/fulldata/sub-OE011/ses-Nordhavn/alldata.csv: 'cum_dist' column not found.
[INFO] Averaged data over 108 intervals using mean() which ignores NaN values.
[INFO] Averaged data over 93 intervals using mean() which ignores NaN values.
[ERROR] Error processing /home/s243636/master-thesis/fulldata/sub-OE023/ses-Nordhavn/alldata.csv: arange: cannot compute length
[INFO] Averaged data over 76 intervals using mean() which ignores NaN values.
[WARN] Skipping /home/s243636/master-thesis/fulldata/sub-OE020/ses-Norrebro/alldata.csv: 'cum_dist' column not found.
[ERROR] Error p

# MLM

In [8]:
def mode_non_empty(x):
    # Remove NaNs, empty strings, and the string "Unknown", then return the mode.
    x = x.dropna()
    x = x[(x != "") & (x.str.strip().str.lower() != "unknown")]
    if not x.empty:
        return x.mode()[0]
    else:
        return np.nan

def average_by_gps(df, dist):
    """
    Averages rows in the DataFrame based on the 'cum_dist' column.
    All rows whose cum_dist falls into the same interval of length 'dist'
    are averaged together.
    
    Numeric columns are averaged,
    and non-numeric columns (e.g., 'typology' and 'sex') are aggregated using the mode.
    
    Rows with missing, empty, or "Unknown" values in non-numeric columns are removed.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame containing a "cum_dist" column (in meters) and other columns.
    dist : float
        Distance interval in meters.
    
    Returns:
    --------
    pd.DataFrame
        A new DataFrame with one row per interval. Numeric columns are averaged,
        and non-numeric columns are aggregated using the mode.
        Also includes an "avg_cum_dist" column that is the mean cum_dist in the interval.
    """
    # For the non-numeric columns we want to preserve, remove rows that are missing, empty, or "Unknown".
    for col in ['typology', 'sex']:
        if col in df.columns:
            df = df[df[col].notna() & (df[col].astype(str).str.strip() != "") & (df[col].astype(str).str.strip().str.lower() != "unknown")]
    
    # If no valid rows remain, return an empty DataFrame.
    if df.empty:
        print("[WARN] No valid rows after filtering for non-missing typology/sex.")
        return pd.DataFrame()
    
    max_dist = df['cum_dist'].max()
    bins = np.arange(0, max_dist + dist, dist)
    labels = (bins[:-1] + bins[1:]) / 2

    df = df.copy()
    df['dist_bin'] = pd.cut(df['cum_dist'], bins=bins, labels=labels, include_lowest=True)
    
    # Compute means for numeric columns.
    avg_df = df.groupby('dist_bin').mean(numeric_only=True).reset_index()
    avg_df.rename(columns={'cum_dist': 'avg_cum_dist'}, inplace=True)
    avg_df['dist_bin'] = avg_df['dist_bin'].astype(float)
    
    # For non-numeric columns, compute the mode for each distance bin.
    for col in ['typology', 'sex']:
        if col in df.columns:
            agg_mode = df.groupby('dist_bin')[col].agg(mode_non_empty).reset_index()
            # Drop bins where mode could not be computed.
            agg_mode = agg_mode.dropna(subset=[col])
            avg_df = pd.merge(avg_df, agg_mode, on='dist_bin', how='inner')
    
    print(f"[INFO] Averaged data over {len(labels)} intervals.")
    return avg_df

def process_all_datasets(parent_dir, dist):
    """
    Process all datasets in a folder tree. For each CSV file found under the folder
    structure, this function reads the CSV, ensures the typology column exists,
    calls average_by_gps, and adds columns extracted from the folder names 
    (e.g., subject code and session).
    
    Parameters:
    -----------
    parent_dir : str
        The parent directory containing the dataset folders.
    dist : float
        The distance interval (in meters) to average over.
    
    Returns:
    --------
    pd.DataFrame
        A design matrix DataFrame where each row corresponds to the average data over a
        distance interval from one session.
    """
    all_dfs = []
    pattern = os.path.join(parent_dir, "sub-*", "ses-*", "alldata.csv")
    file_list = glob.glob(pattern, recursive=True)
    print(f"[INFO] Found {len(file_list)} dataset files.")
    
    for file_path in file_list:
        try:
            df = pd.read_csv(file_path)
            # Ensure 'typology' is present; if not, add it (these rows will be filtered later).
            if 'typology' not in df.columns:
                df['typology'] = "Unknown"
            # Similarly, if 'sex' is missing, add it.
            if 'sex' not in df.columns:
                df['sex'] = "Unknown"
                
            if 'cum_dist' not in df.columns:
                print(f"[WARN] Skipping {file_path}: 'cum_dist' column not found.")
                continue

            avg_df = average_by_gps(df, dist)
            if avg_df.empty:
                print(f"[WARN] No valid data after averaging for {file_path}. Skipping.")
                continue
            # Extract subject code and session from the file path.
            m = re.search(r"sub-([A-Z0-9]+)", file_path, re.IGNORECASE)
            subject_code = m.group(1) if m else "Unknown"
            m2 = re.search(r"ses-([A-Za-z0-9]+)", file_path)
            session = m2.group(1) if m2 else "Unknown"
            avg_df['subject_code'] = subject_code
            avg_df['session'] = session
            
            # Preserve participant_id if present; otherwise use subject_code as proxy.
            if 'participant_id' not in avg_df.columns:
                avg_df['participant_id'] = subject_code

            avg_df['source_file'] = os.path.basename(file_path)
            all_dfs.append(avg_df)
        except Exception as e:
            print(f"[ERROR] Error processing {file_path}: {e}")
    
    if not all_dfs:
        print("[ERROR] No data processed.")
        return None
    design_df = pd.concat(all_dfs, ignore_index=True)
    print(f"[INFO] Design matrix has {design_df.shape[0]} rows and {design_df.shape[1]} columns.")
    return design_df

def fit_mixedlm(design_df, response_var, predictors, group_var='participant_id'):
    """
    Fits a Linear Mixed Effects Model to the design matrix.
    
    Parameters:
    -----------
    design_df : pd.DataFrame
        The design matrix DataFrame with averaged rows.
    response_var : str
        The name of the dependent variable in the design matrix.
    predictors : list of str
        A list of column names to use as independent variables.
        (If 'typology' is one of them, it will be automatically converted to a categorical predictor.)
    group_var : str
        The column name representing the grouping variable for random effects.
    
    Returns:
    --------
    MixedLMResults
        The fitted mixed effects model results.
    """
    # Check for missing values in the required columns.
    cols_to_check = [response_var] + predictors + [group_var]
    total_rows = design_df.shape[0]
    print("[DEBUG] Missing values for MixedLM columns:")
    for col in cols_to_check:
        if col in design_df.columns:
            n_missing = design_df[col].isna().sum()
            pct_missing = (n_missing / total_rows) * 100
            print(f"  {col}: {n_missing} missing out of {total_rows} rows ({pct_missing:.1f}%)")
        else:
            print(f"  {col}: Column not found!")
    
    design_df_clean = design_df.dropna(subset=cols_to_check)
    print(f"[INFO] MixedLM: {design_df_clean.shape[0]} rows remain after dropping missing values.")
    
    if design_df_clean.shape[0] == 0:
        raise ValueError("No rows remain after dropping missing values; please check the input columns.")
    
    # Convert 'typology' predictor to categorical if it is in the predictors list.
    new_predictors = []
    for p in predictors:
        if p.lower() == 'typology':
            new_predictors.append("C(typology)")
        else:
            new_predictors.append(p)
    fixed_effects = " + ".join(new_predictors)
    formula = f'Q("{response_var}") ~ {fixed_effects}'
    print(f"[INFO] MixedLM formula: {formula}")
    
    md = smf.mixedlm(formula, design_df_clean, groups=design_df_clean[group_var])
    mdf = md.fit()
    print(mdf.summary())
    
    plt.figure(figsize=(8, 6))
    plt.scatter(mdf.fittedvalues, mdf.resid, alpha=0.7, edgecolor='k')
    plt.xlabel("Fitted Values", fontsize=12)
    plt.ylabel("Residuals", fontsize=12)
    plt.title("MixedLM: Residual vs. Fitted Values", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    return mdf

# Example usage:
if __name__ == '__main__':
    parent_directory = os.path.join(workdir, 'fulldata')
    distance_interval = 10
    design_matrix = process_all_datasets(parent_directory, distance_interval)
    
    # Define response and predictor columns.
    response = 'eda_raw'
    predictors = [
        'age', 'sex',
        'typology',
        'wind_speed', 'noise_level', 'humidity', 'pm2_5', 'pm10_0', 'temp_atmospheric',
    ]    
    # Print which predictors will be used.
    available_predictors = [p for p in predictors if p in design_matrix.columns or p.lower() in [c.lower() for c in design_matrix.columns]]
    print(f"[INFO] Using predictors: {available_predictors}")
    
    # For mixed models, ensure that the design matrix includes the 'participant_id' column.
    if response in design_matrix.columns and available_predictors and 'participant_id' in design_matrix.columns:
        mixedlm_results = fit_mixedlm(design_matrix, response, available_predictors, group_var='participant_id')
    else:
        print("[ERROR] The required columns for the mixed effects model are not present in the design matrix.")

[INFO] Found 25 dataset files.
[WARN] No valid rows after filtering for non-missing typology/sex.
[WARN] No valid data after averaging for /home/s243636/master-thesis/fulldata/sub-OE004/ses-Norreport/alldata.csv. Skipping.
[WARN] No valid rows after filtering for non-missing typology/sex.
[WARN] No valid data after averaging for /home/s243636/master-thesis/fulldata/sub-OE018/ses-Hellerup/alldata.csv. Skipping.
[WARN] Skipping /home/s243636/master-thesis/fulldata/sub-OE024/ses-Nordhavn/alldata.csv: 'cum_dist' column not found.
[WARN] No valid rows after filtering for non-missing typology/sex.
[WARN] No valid data after averaging for /home/s243636/master-thesis/fulldata/sub-OE019/ses-Hellerup/alldata.csv. Skipping.
[WARN] Skipping /home/s243636/master-thesis/fulldata/sub-OE011/ses-Nordhavn/alldata.csv: 'cum_dist' column not found.
[WARN] No valid rows after filtering for non-missing typology/sex.
[WARN] No valid data after averaging for /home/s243636/master-thesis/fulldata/sub-OE023/ses-

AttributeError: 'NoneType' object has no attribute 'columns'

In [ ]:
# reference is GARDEN / PARK
print(design_matrix['typology'].unique())


# Typology Analysis

In [ ]:
def process_all_datasets(parent_dir, dist):
    """
    Process all datasets in a folder tree. For each CSV file found under the folder
    structure, this function reads the CSV, calls average_by_gps, and adds columns 
    extracted from the folder names (e.g., subject code and session).
    
    Parameters:
    -----------
    parent_dir : str
        The parent directory containing the dataset folders.
    dist : float
        The distance interval (in meters) to average over.
    
    Returns:
    --------
    pd.DataFrame
        A design matrix DataFrame where each row corresponds to the average data over a
        distance interval from one session.
    """
    all_dfs = []
    pattern = os.path.join(parent_dir, "sub-*", "ses-*", "alldata.csv")
    file_list = glob.glob(pattern, recursive=True)
    print(f"[INFO] Found {len(file_list)} dataset files.")
    
    for file_path in file_list:
        try:
            df = pd.read_csv(file_path)
            if 'cum_dist' not in df.columns:
                print(f"[WARN] Skipping {file_path}: 'cum_dist' column not found.")
                continue

            avg_df = average_by_gps(df, dist)
            # Extract subject code; you can modify this if 'participant_id' is present in the CSV.
            m = re.search(r"sub-([A-Z0-9]+)", file_path, re.IGNORECASE)
            subject_code = m.group(1) if m else "Unknown"
            m2 = re.search(r"ses-([A-Za-z0-9]+)", file_path)
            session = m2.group(1) if m2 else "Unknown"
            avg_df['subject_code'] = subject_code
            avg_df['session'] = session
            
            # If the CSV files already include a 'participant_id', it will be preserved.
            # Otherwise, you can use the extracted subject_code as a proxy.
            if 'participant_id' not in avg_df.columns:
                avg_df['participant_id'] = subject_code

            avg_df['source_file'] = os.path.basename(file_path)
            all_dfs.append(avg_df)
        except Exception as e:
            print(f"[ERROR] Error processing {file_path}: {e}")
    
    if not all_dfs:
        print("[ERROR] No data processed.")
        return None
    design_df = pd.concat(all_dfs, ignore_index=True)
    print(f"[INFO] Design matrix has {design_df.shape[0]} rows and {design_df.shape[1]} columns.")
    return design_df

def compute_typology_averages(design_df, interest_var):
    """
    Compute the average of the interest variable for each typology for each participant.
    
    Parameters:
    -----------
    design_df : pd.DataFrame
        The full data table containing at least the columns: 'participant_id', 'typology', and interest_var.
    interest_var : str
        The column name for the variable of interest.
        
    Returns:
    --------
    pd.DataFrame
        A new DataFrame with one row per participant per typology containing the mean value.
    """
    # Group by participant and typology, computing the mean of the interest variable.
    avg_df = design_df.groupby(['participant_id', 'typology'])[interest_var].mean().reset_index()
    avg_df.rename(columns={interest_var: f'avg_{interest_var}'}, inplace=True)
    return avg_df

def run_repeated_measures_anova(avg_df, interest_var):
    """
    Perform a repeated measures ANOVA on the aggregated data.
    
    Parameters:
    -----------
    avg_df : pd.DataFrame
        DataFrame containing columns: 'participant_id', 'typology', and the averaged interest variable.
    interest_var : str
        The original name of the interest variable.
        
    Returns:
    --------
    AnovaRM result
    """
    # Rename the column for clarity in the ANOVA
    depvar = f'avg_{interest_var}'
    
    # Check if each participant has at least one measurement for each typology.
    typology_counts = avg_df.groupby('participant_id')['typology'].nunique()
    if (typology_counts < avg_df['typology'].nunique()).any():
        print("[WARNING] Some participants do not have data for all typologies. Consider using a mixed-effects model instead.")
    
    # Run repeated measures ANOVA using statsmodels' AnovaRM
    aovrm = AnovaRM(avg_df, depvar=depvar, subject='participant_id', within=['typology'])
    anova_results = aovrm.fit()
    print(anova_results)
    return anova_results

def plot_typology_differences(avg_df, interest_var):
    """
    Create a box plot to visualize differences in the interest variable across typologies.
    
    Parameters:
    -----------
    avg_df : pd.DataFrame
        DataFrame containing per-participant averages for each typology.
    interest_var : str
        The original name of the interest variable.
    """
    depvar = f'avg_{interest_var}'
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='typology', y=depvar, data=avg_df)
    plt.xlabel("Typology")
    plt.ylabel(f"Average {interest_var}")
    plt.title(f"Distribution of Average {interest_var} by Typology")
    plt.tight_layout()
    plt.show()

# Example usage:
if __name__ == '__main__':
    parent_directory = os.path.join(workdir, 'fulldata')
    distance_interval = 10
    design_matrix = process_all_datasets(parent_directory, distance_interval)
    # 'participant_id', 'typology', and the interest variable (e.g., 'frontal alpha asymmetry').
    interest_variable = 'frontal alpha asymmetry'  # or any other variable of interest
    # Compute per-participant, per-typology averages.
    avg_typo_df = compute_typology_averages(design_matrix, interest_variable)
    
    # Run inferential statistics using repeated measures ANOVA.
    anova_results = run_repeated_measures_anova(avg_typo_df, interest_variable)
    
    # Plot the differences.
    plot_typology_differences(avg_typo_df, interest_variable)
